In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (files are in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')

# Convert timestamp (seconds) to timezone-aware datetimes in Asia/Shanghai
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# --- metric.csv summary ---
metric_row_count = df_metric.shape[0]
metric_ts_min = df_metric['ts'].min()
metric_ts_max = df_metric['ts'].max()
metric_cmdbs = sorted(df_metric['cmdb_id'].astype(str).unique())
metric_kpi_counts = df_metric['kpi_name'].value_counts()
metric_top20 = metric_kpi_counts.head(20).reset_index()
metric_top20.columns = ['kpi_name', 'count']
metric_kpi_distinct = metric_kpi_counts.shape[0]

metric_summary = pd.DataFrame([{
    'file': 'metric.csv',
    'rows': metric_row_count,
    'ts_min': metric_ts_min.isoformat() if pd.notnull(metric_ts_min) else None,
    'ts_max': metric_ts_max.isoformat() if pd.notnull(metric_ts_max) else None,
    'distinct_cmdb_ids': ','.join(metric_cmdbs),
    'distinct_kpi_count': metric_kpi_distinct
}])

# --- trace.csv summary ---
trace_row_count = df_trace.shape[0]
trace_ts_min = df_trace['ts'].min()
trace_ts_max = df_trace['ts'].max()
trace_cmdbs = sorted(df_trace['cmdb_id'].astype(str).unique())
trace_name_counts = df_trace['trace_name'].value_counts()
trace_top20 = trace_name_counts.head(20).reset_index()
trace_top20.columns = ['trace_name', 'count']
trace_name_distinct = trace_name_counts.shape[0]

trace_summary = pd.DataFrame([{
    'file': 'trace.csv',
    'rows': trace_row_count,
    'ts_min': trace_ts_min.isoformat() if pd.notnull(trace_ts_min) else None,
    'ts_max': trace_ts_max.isoformat() if pd.notnull(trace_ts_max) else None,
    'distinct_cmdb_ids': ','.join(trace_cmdbs),
    'distinct_trace_count': trace_name_distinct
}])

# Display compact summaries and top-20 name counts (bounded)
metric_summary, trace_summary, metric_top20.head(20), trace_top20.head(20)

```
Out[1]:
```
```python
# Compose a concise plain-English summary based on previous execution results stored in variables.
m = metric_summary.loc[0]
t = trace_summary.loc[0]
m_top = metric_top20.head(10)
t_top = trace_top20.head(10)

m_top_str = ', '.join(m_top.apply(lambda r: f"{r['kpi_name']}({int(r['count'])})", axis=1))
t_top_str = ', '.join(t_top.apply(lambda r: f"{r['trace_name']}({int(r['count'])})", axis=1))

summary = (
    f"metric.csv: {int(m['rows'])} rows; time range {m['ts_min']} to {m['ts_max']}; "
    f"{len(m['distinct_cmdb_ids'].split(','))} distinct cmdb_id values ({m['distinct_cmdb_ids']}); "
    f"{int(m['distinct_kpi_count'])} distinct KPI names. Top KPIs (top 10 shown): {m_top_str}.\n\n"
    f"trace.csv: {int(t['rows'])} rows; time range {t['ts_min']} to {t['ts_max']}; "
    f"{len(t['distinct_cmdb_ids'].split(','))} distinct cmdb_id values ({t['distinct_cmdb_ids']}); "
    f"{int(t['distinct_trace_count'])} distinct trace names. Top trace names (top 10 shown): {t_top_str}."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file   rows                     ts_min                     ts_max                                  distinct_cmdb_ids  distinct_kpi_count
0  metric.csv  90998  2020-04-11T01:30:00+08:00  2020-04-11T02:30:00+08:00  db_001,db_002,db_003,db_004,db_005,db_006,db_0...                 131,         file  rows                     ts_min                     ts_max                                  distinct_cmdb_ids  distinct_trace_count
0  trace.csv  6240  2020-04-11T02:00:00+08:00  2020-04-11T02:29:00+08:00  docker_001,docker_002,docker_003,docker_004,do...                    76,                     kpi_name  count
0                 Send_total   1323
1               Disk_io_util   1322
2   Incoming_network_traffic   1321
3       Processor_load_5_min   1321
4                Disk_rd_ios   1321
5   Outgoing_network_traffic   1321
6                Memory_used   1320
7                   ss_total   1320
8                    Page_pi   1320
9                 Disk_svctm   1320
10               Disk_wr_kbs   1320
11           Memory_used_pct   1320
12     Processor_load_15_min   1320
13                Disk_await   1320
14             CPU_user_time   1320
15             Disk_avgqu_sz   1320
16          Memory_available   1320
17                   Page_po   1319
18              CPU_idle_pct   1319
19                Agent_ping   1319,                                        trace_name  count
0                       trace.self.jdbc.row_count    240
1                      trace.self.jdbc.error_rate    240
2                    trace.self.jdbc.duration_p95    240
3                   trace.self.jdbc.duration_mean    240
4                      trace.self.local.row_count    240
5                     trace.self.local.error_rate    240
6                   trace.self.local.duration_p95    240
7                  trace.self.local.duration_mean    240
8                        trace.self.csf.row_count    180
9                       trace.self.csf.error_rate    180
10                    trace.self.csf.duration_p95    180
11                   trace.self.csf.duration_mean    180
12                 trace.self.flyremote.row_count    120
13                trace.self.flyremote.error_rate    120
14              trace.self.flyremote.duration_p95    120
15             trace.self.flyremote.duration_mean    120
16  trace.from_os_022.remoteprocess.duration_mean     60
17   trace.from_os_022.remoteprocess.duration_p95     60
18     trace.from_os_022.remoteprocess.error_rate     60
19      trace.from_os_022.remoteprocess.row_count     60)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# ensure timezone variable
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 for each (cmdb_id, kpi_name) using entire series (global thresholds)
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(P5=lambda s: s.quantile(0.05), P95=lambda s: s.quantile(0.95))
    .reset_index()
)

# 2) Filter metric.csv to the incident window 2020-04-11 02:00:00+08 through 2020-04-11 02:30:00+08 (inclusive)
start = pd.Timestamp('2020-04-11 02:00:00').tz_localize(tz)
end = pd.Timestamp('2020-04-11 02:30:00').tz_localize(tz)
mask_window = (df_metric['ts'] >= start) & (df_metric['ts'] <= end)
df_window = df_metric[mask_window].copy()

# 3) Mark anomalies in the window using the precomputed global thresholds
df_w = df_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
df_w['is_anomaly'] = (df_w['value'] >= df_w['P95']) | (df_w['value'] <= df_w['P5'])

# 4) Aggregate anomalies by (cmdb_id, kpi_name)
# anomaly counts and earliest anomaly timestamp (within window)
anom_agg = (
    df_w[df_w['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(anomaly_count=('is_anomaly', 'sum'),
         earliest_anomaly_timestamp=('ts', 'min'))
    .reset_index()
)

# max/min values in the window (full window, not only anomaly points)
window_stats = (
    df_w
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(max_value_in_window=('value', 'max'),
         min_value_in_window=('value', 'min'))
    .reset_index()
)

# combine with global thresholds
anom_combined = (
    anom_agg
    .merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(window_stats, on=['cmdb_id', 'kpi_name'], how='left')
)

# format earliest timestamp as ISO string for compact display
anom_combined['earliest_anomaly_timestamp'] = anom_combined['earliest_anomaly_timestamp'].dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

# order by anomaly_count desc and keep top 20
anomalies_top20 = (
    anom_combined
    .sort_values(['anomaly_count', 'earliest_anomaly_timestamp'], ascending=[False, True])
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'P95', 'P5', 'max_value_in_window', 'min_value_in_window']]
    .head(20)
    .reset_index(drop=True)
)

anomalies_top20

```
Out[2]:
```
# Compose a concise plain-English summary of the anomaly aggregation stored in `anomalies_top20`.
# Use the existing `anomalies_top20` variable from the kernel and produce a single summary string.
topn = anomalies_top20.shape[0]
rows = anomalies_top20.head(5).to_dict(orient='records')  # summarize top 5 for compactness

lines = [
    f"Top {topn} component-KPI pairs by anomaly count (window 2020-04-11 02:00 to 2020-04-11 02:30 +08):"
]
for i, r in enumerate(rows, 1):
    lines.append(
        f"{i}. {r['cmdb_id']} / {r['kpi_name']}: "
        f"anomalies={int(r['anomaly_count'])}, earliest={r['earliest_anomaly_timestamp']}, "
        f"P95={r['P95']}, P5={r['P5']}, max_in_window={r['max_value_in_window']}, min_in_window={r['min_value_in_window']}"
    )

if topn > 5:
    lines.append(f"...plus {topn-5} more rows (output limited to top {topn}).")

summary = " ".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id             kpi_name  anomaly_count earliest_anomaly_timestamp         P95          P5  max_value_in_window  min_value_in_window
0      os_007          Disk_rd_ios             31   2020-04-11T02:00:00+0800    0.000000    0.000000             0.000000             0.000000
1      os_020         Disk_io_util             31   2020-04-11T02:00:00+0800    0.000000    0.000000             0.000000             0.000000
2   redis_011           redis_ping             31   2020-04-11T02:00:00+0800    1.000000    1.000000             1.000000             1.000000
3   redis_012           redis_load             31   2020-04-11T02:00:00+0800    0.000000    0.000000             0.000000             0.000000
4      db_001                  ACS             30   2020-04-11T02:00:00+0800    0.020000    0.010000             0.020000             0.000000
5      db_001                 AIOS             30   2020-04-11T02:00:00+0800    0.010000    0.000000             0.010000             0.000000
6      db_001                 Hang             30   2020-04-11T02:00:00+0800    1.000000    1.000000             1.000000             1.000000
7      db_001            MEM_Total             30   2020-04-11T02:00:00+0800  381.902264  381.902264           381.902264           381.902264
8      db_001        MEM_real_util             30   2020-04-11T02:00:00+0800   46.166000   46.150000            46.280000            46.150000
9      db_001         On_Off_State             30   2020-04-11T02:00:00+0800    1.000000    1.000000             1.000000             1.000000
10     db_001        Proc_Used_Pct             30   2020-04-11T02:00:00+0800    1.650000    1.640000             1.650000             1.640000
11     db_001   Proc_User_Used_Pct             30   2020-04-11T02:00:00+0800    1.650000    1.640000             1.650000             1.640000
12     db_001             Row_Lock             30   2020-04-11T02:00:00+0800    0.000000    0.000000             0.000000             0.000000
13     db_001      SctRead_Per_Sec             30   2020-04-11T02:00:00+0800    0.000000    0.000000             0.000000             0.000000
14     db_001  tnsping_result_time             30   2020-04-11T02:00:00+0800   10.000000    0.000000            10.000000             0.000000
15     db_002                  ACS             30   2020-04-11T02:00:00+0800    0.020000    0.010000             0.020000             0.010000
16     db_002                 AIOS             30   2020-04-11T02:00:00+0800    0.000000    0.000000             0.000000             0.000000
17     db_002                 Hang             30   2020-04-11T02:00:00+0800    1.000000    1.000000             1.000000             1.000000
18     db_002        Login_Per_Sec             30   2020-04-11T02:00:00+0800    0.400000    0.300000             0.500000             0.300000
19     db_002            MEM_Total             30   2020-04-11T02:00:00+0800  381.902264  381.902264           381.902264           381.902264```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# ensure required kernel variables exist; otherwise load minimally
try:
    df_w
except NameError:
    # load metric.csv and recompute windowed merged df_w and thresholds if needed
    df_metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # global thresholds
    thresholds = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(P5=lambda s: s.quantile(0.05), P95=lambda s: s.quantile(0.95))
        .reset_index()
    )
    # window
    start = pd.Timestamp('2020-04-11 02:00:00').tz_localize(tz)
    end = pd.Timestamp('2020-04-11 02:30:00').tz_localize(tz)
    mask_window = (df_metric['ts'] >= start) & (df_metric['ts'] <= end)
    df_window = df_metric[mask_window].copy()
    df_w = df_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# ensure anomalies_top20 exists; if not, compute minimal placeholder (empty)
try:
    anomalies_top20
except NameError:
    anomalies_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name'])

# timezone
tz = pytz.timezone('Asia/Shanghai')

# 1) take top 10 rows from anomalies_top20 (ranked)
top10 = anomalies_top20.head(10).loc[:, ['cmdb_id', 'kpi_name']].drop_duplicates()

# 1a) extract filtered-window points for these pairs from df_w
df_top = df_w.merge(top10, on=['cmdb_id', 'kpi_name'], how='inner').copy()

# If no matching rows, return empty table with specified columns
out_cols = [
    'cmdb_id','kpi_name','anomaly_type','segment_start_timestamp','segment_end_timestamp',
    'segment_length_minutes','max_value_in_segment','min_value_in_segment','severity_value'
]

if df_top.shape[0] == 0:
    segments_result = pd.DataFrame(columns=out_cols)
else:
    # 1b) mark each timestamp as 'high' or 'low' anomaly using robust pandas operations
    df_top['P95'] = pd.to_numeric(df_top['P95'], errors='coerce')
    df_top['P5'] = pd.to_numeric(df_top['P5'], errors='coerce')
    df_top['anomaly_type'] = pd.NA  # use pandas NA for object-like behavior

    # set high and low via .loc to avoid dtype promotion issues
    df_top.loc[df_top['value'] >= df_top['P95'], 'anomaly_type'] = 'high'
    df_top.loc[df_top['value'] <= df_top['P5'], 'anomaly_type'] = 'low'

    # keep only anomaly points
    df_top_anom = df_top[df_top['anomaly_type'].notna()].copy()

    if df_top_anom.shape[0] == 0:
        segments_result = pd.DataFrame(columns=out_cols)
    else:
        # 1c) find consecutive anomaly segments (timestamps differ by exactly 60 seconds)
        df_top_anom = df_top_anom.sort_values(['cmdb_id','kpi_name','anomaly_type','timestamp']).reset_index(drop=True)

        # compute timestamp diff in seconds within each (cmdb,kpi,anomaly_type)
        df_top_anom['ts_diff'] = df_top_anom.groupby(['cmdb_id','kpi_name','anomaly_type'])['timestamp'].diff().fillna(0)

        # new segment when diff != 60
        df_top_anom['new_seg_flag'] = (df_top_anom['ts_diff'] != 60).astype(int)

        # cumulative sum of new_seg_flag within group gives segment id
        df_top_anom['seg_seq'] = df_top_anom.groupby(['cmdb_id','kpi_name','anomaly_type'])['new_seg_flag'].cumsum()

        # aggregate per segment
        seg_agg = (
            df_top_anom
            .groupby(['cmdb_id','kpi_name','anomaly_type','seg_seq'], as_index=False)
            .agg(
                segment_start_ts=('timestamp', 'min'),
                segment_end_ts=('timestamp', 'max'),
                segment_length_minutes=('timestamp', 'count'),
                max_value_in_segment=('value', 'max'),
                min_value_in_segment=('value', 'min'),
                P95=('P95', 'first'),
                P5=('P5', 'first')
            )
        )

        # compute severity_value
        seg_agg['severity_value'] = np.where(
            seg_agg['anomaly_type'] == 'high',
            seg_agg['max_value_in_segment'] - seg_agg['P95'],
            seg_agg['P5'] - seg_agg['min_value_in_segment']
        )

        # convert start/end timestamps (seconds) to Asia/Shanghai ISO strings
        seg_agg['segment_start_timestamp'] = pd.to_datetime(seg_agg['segment_start_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')
        seg_agg['segment_end_timestamp'] = pd.to_datetime(seg_agg['segment_end_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

        # 2) filter out segments of length < 3 minutes
        seg_filtered = seg_agg[seg_agg['segment_length_minutes'] >= 3].copy()

        # 3) sort and limit to top 50
        segments_result = (
            seg_filtered
            .sort_values(['segment_length_minutes', 'severity_value'], ascending=[False, False])
            .loc[:, ['cmdb_id','kpi_name','anomaly_type','segment_start_timestamp','segment_end_timestamp',
                     'segment_length_minutes','max_value_in_segment','min_value_in_segment','severity_value']]
            .head(50)
            .reset_index(drop=True)
        )

# final compact table
segments_result

```
Out[3]:
```
Summary of detected anomaly segments (window 2020-04-11 02:00 → 02:30, Asia/Shanghai):

- Total segments returned (length ≥ 3 minutes) from the top10 component-KPI pairs: 18.
- Many long continuous "low" segments span the entire window (31 minutes) for:
  1) os_007 / Disk_rd_ios — low, 02:00→02:30, 31 minutes
  2) os_020 / Disk_io_util — low, 02:00→02:30, 31 minutes
  3) redis_011 / redis_ping — low, 02:00→02:30, 31 minutes
  4) redis_012 / redis_load — low, 02:00→02:30, 31 minutes
- db_001 has multiple long segments (mostly 30 or 20+ minutes) across KPIs such as Hang, MEM_Total, On_Off_State, and MEM_real_util.
- ACS (db_001) shows multiple shorter segments (3–5 minutes) of both high and low anomalies.
- Severity values are almost all 0.00 for these segments (meaning observed values equal the stored global P95 or P5); one segment shows a small non-zero severity (ACS low, 3 minutes, severity 0.01).
- Interpretation: several metrics remained continuously at or beyond their global thresholds through the incident window (many exactly equal to the threshold), producing long multi-minute anomaly segments rather than isolated spikes.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id       kpi_name anomaly_type   segment_start_timestamp     segment_end_timestamp  segment_length_minutes  max_value_in_segment  min_value_in_segment  severity_value
0      os_007    Disk_rd_ios          low  2020-04-11T02:00:00+0800  2020-04-11T02:30:00+0800                      31              0.000000              0.000000            0.00
1      os_020   Disk_io_util          low  2020-04-11T02:00:00+0800  2020-04-11T02:30:00+0800                      31              0.000000              0.000000            0.00
2   redis_011     redis_ping          low  2020-04-11T02:00:00+0800  2020-04-11T02:30:00+0800                      31              1.000000              1.000000            0.00
3   redis_012     redis_load          low  2020-04-11T02:00:00+0800  2020-04-11T02:30:00+0800                      31              0.000000              0.000000            0.00
4      db_001           Hang          low  2020-04-11T02:00:00+0800  2020-04-11T02:29:00+0800                      30              1.000000              1.000000            0.00
5      db_001      MEM_Total          low  2020-04-11T02:00:00+0800  2020-04-11T02:29:00+0800                      30            381.902264            381.902264            0.00
6      db_001   On_Off_State          low  2020-04-11T02:00:00+0800  2020-04-11T02:29:00+0800                      30              1.000000              1.000000            0.00
7      db_001  MEM_real_util          low  2020-04-11T02:10:00+0800  2020-04-11T02:29:00+0800                      20             46.150000             46.150000            0.00
8      db_001           AIOS          low  2020-04-11T02:09:00+0800  2020-04-11T02:17:00+0800                       9              0.000000              0.000000            0.00
9      db_001           AIOS          low  2020-04-11T02:19:00+0800  2020-04-11T02:27:00+0800                       9              0.000000              0.000000            0.00
10     db_001  MEM_real_util          low  2020-04-11T02:00:00+0800  2020-04-11T02:08:00+0800                       9             46.150000             46.150000            0.00
11     db_001           AIOS          low  2020-04-11T02:00:00+0800  2020-04-11T02:07:00+0800                       8              0.000000              0.000000            0.00
12     db_001            ACS         high  2020-04-11T02:02:00+0800  2020-04-11T02:06:00+0800                       5              0.020000              0.020000            0.00
13     db_001            ACS          low  2020-04-11T02:15:00+0800  2020-04-11T02:19:00+0800                       5              0.010000              0.010000            0.00
14     db_001            ACS          low  2020-04-11T02:27:00+0800  2020-04-11T02:29:00+0800                       3              0.010000              0.000000            0.01
15     db_001            ACS         high  2020-04-11T02:12:00+0800  2020-04-11T02:14:00+0800                       3              0.020000              0.020000            0.00
16     db_001            ACS         high  2020-04-11T02:22:00+0800  2020-04-11T02:24:00+0800                       3              0.020000              0.020000            0.00
17     db_001            ACS          low  2020-04-11T02:07:00+0800  2020-04-11T02:09:00+0800                       3              0.010000              0.010000            0.00```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse existing df_trace if present; otherwise load and prepare
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# ensure tz variable
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 for each (cmdb_id, trace_name) using the full series
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(P95=lambda s: s.quantile(0.95), P5=lambda s: s.quantile(0.05))
    .reset_index()
)

# 2) Filter trace.csv to the incident window (inclusive)
start = pd.Timestamp('2020-04-11 02:00:00').tz_localize(tz)
end = pd.Timestamp('2020-04-11 02:30:00').tz_localize(tz)
mask_window = (df_trace['ts'] >= start) & (df_trace['ts'] <= end)
df_trace_window = df_trace[mask_window].copy()

# 3) Mark anomalies using precomputed global thresholds
df_tw = df_trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
df_tw['is_anomaly'] = (df_tw['value'] >= df_tw['P95']) | (df_tw['value'] <= df_tw['P5'])

# 4) Aggregate anomalies by (cmdb_id, trace_name)
anom_groups = df_tw[df_tw['is_anomaly']].groupby(['cmdb_id', 'trace_name'])

trace_anom_agg = (
    anom_groups
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('ts', 'min'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min')
    )
    .reset_index()
    .merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
)

# format earliest timestamp as ISO string in Asia/Shanghai
trace_anom_agg['earliest_anomaly_timestamp'] = trace_anom_agg['earliest_anomaly_timestamp'].dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')

# order by anomaly_count desc and keep top 20, select requested columns
trace_anom_top20 = (
    trace_anom_agg
    .sort_values('anomaly_count', ascending=False)
    .loc[:, ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'P95', 'P5', 'max_value_in_window', 'min_value_in_window']]
    .head(20)
    .reset_index(drop=True)
)

trace_anom_top20

```
Out[4]:
```
Summary of trace anomalies in the window 2020-04-11 02:00→02:30 (+08):

- Top pairs: the returned top 20 (ranked by anomaly_count) are dominated by docker containers (docker_001, docker_002, docker_003, docker_007, docker_002, docker_003, etc.). Examples include:
  - docker_001 / trace.from_os_022.remoteprocess.error_rate — anomaly_count 30, earliest 2020-04-11T02:00:00+08, P95=0.000, P5=0.000, max=0.000, min=0.000
  - docker_001 / trace.self.csf.error_rate — anomaly_count 30, earliest 02:00, P95=0.000, P5=0.000, max=0.000, min=0.000
  - docker_001 / trace.self.jdbc.duration_p95 — anomaly_count 30, earliest 02:00, P95=0.003, P5=0.001, max=0.004, min=0.001
  - docker_002 / trace.self.jdbc.error_rate — anomaly_count 30, earliest 02:00, P95≈0.000466, P5=0.000, max≈0.000943, min=0.000
  - docker_007 / trace.self.jdbc.duration_p95 — anomaly_count 30, earliest 02:00, P95=0.003, P5=0.003, max=0.003, min=0.003

- Common pattern:
  - Every listed pair has anomaly_count = 30 (i.e., an anomaly every minute in the 30-minute window), with earliest anomaly at 02:00:00+08.
  - Many trace series have P95 and P5 equal to 0 and max/min = 0, indicating constant-zero values (which triggered low-or-high threshold matches).
  - A few trace series (e.g., some trace.self.jdbc.duration_p95 and trace.self.jdbc.error_rate) show small non-zero P95/P5 and small variation between max and min.

- Interpretation / immediate notes:
  - Anomalies are widespread and continuous across multiple docker-hosted trace metrics during the entire incident window.
  - For many series the values are constant (often zero), causing every point to meet the threshold condition; this could indicate either persistent condition at threshold or missing/flatlined telemetry.
  - For duration/error-rate traces with small non-zero P95/P5, there is minor variation but still every point fell outside the global P5–P95 band.

- Suggested next steps:
  - Inspect the affected docker nodes' trace collection and pipeline (to rule out missing/flatlined data).
  - Focus on the JDBC duration/error-rate traces (non-zero values) if investigating performance impact, and check related logs/containers for errors during 02:00–02:30.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    trace_name  anomaly_count earliest_anomaly_timestamp       P95     P5  max_value_in_window  min_value_in_window
0   docker_001    trace.from_os_022.remoteprocess.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
1   docker_001                     trace.self.csf.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
2   docker_001                  trace.self.jdbc.duration_p95             30   2020-04-11T02:00:00+0800  0.003000  0.001             0.004000                0.001
3   docker_001               trace.self.flyremote.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
4   docker_001                    trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.002705                0.000
5   docker_003                    trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.001311                0.000
6   docker_003                  trace.self.jdbc.duration_p95             30   2020-04-11T02:00:00+0800  0.003000  0.001             0.004000                0.001
7   docker_003               trace.self.flyremote.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
8   docker_002  trace.to_docker_007.remoteprocess.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
9   docker_002  trace.to_docker_008.remoteprocess.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
10  docker_002                    trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800  0.000466  0.000             0.000943                0.000
11  docker_002               trace.self.flyremote.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
12  docker_002                     trace.self.csf.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
13  docker_002    trace.from_os_022.remoteprocess.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
14  docker_001  trace.to_docker_008.remoteprocess.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
15  docker_001  trace.to_docker_007.remoteprocess.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
16  docker_003    trace.from_os_021.remoteprocess.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
17  docker_003                     trace.self.csf.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
18  docker_003  trace.to_docker_005.remoteprocess.error_rate             30   2020-04-11T02:00:00+0800  0.000000  0.000             0.000000                0.000
19  docker_007                  trace.self.jdbc.duration_p95             30   2020-04-11T02:00:00+0800  0.003000  0.003             0.003000                0.003```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# 1) Detect in-memory variables for the three sources
g = globals()

# possible names
metric_agg_candidates = ['anomalies_top20', 'anom_combined', 'metric_anom_agg', 'metric_anomalies_top20']
segments_candidates = ['segments_result', 'metric_segments', 'segments', 'metric_anom_segments']
trace_agg_candidates = ['trace_anom_top20', 'trace_anom_agg', 'trace_anomalies_top20', 'trace_anom_agg']

def find_first_existing(cands):
    for name in cands:
        if name in g:
            return g[name]
    return None

metric_agg = find_first_existing(metric_agg_candidates)
segments_df = find_first_existing(segments_candidates)
trace_agg = find_first_existing(trace_agg_candidates)

# Ensure DataFrames exist (otherwise create empty ones with expected cols)
if metric_agg is None:
    metric_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','P95','P5','max_value_in_window','min_value_in_window'])
if segments_df is None:
    segments_df = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_type','segment_start_timestamp','segment_end_timestamp','segment_length_minutes','max_value_in_segment','min_value_in_segment','severity_value'])
if trace_agg is None:
    trace_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','P95','P5','max_value_in_window','min_value_in_window'])

# Make copies to avoid mutating originals
metric_agg = metric_agg.copy()
segments_df = segments_df.copy()
trace_agg = trace_agg.copy()

# Normalize timestamp columns to timezone-aware datetimes (Asia/Shanghai)
def parse_tscol(df, col):
    if col in df.columns and not df[col].empty:
        # to_datetime will parse timezone offset if present; coerce errors
        return pd.to_datetime(df[col], utc=True, errors='coerce').dt.tz_convert(tz)
    else:
        return pd.Series(pd.NaT, index=df.index)

# metric aggregate earliest
if 'earliest_anomaly_timestamp' in metric_agg.columns:
    metric_agg['earliest_ts_dt'] = parse_tscol(metric_agg, 'earliest_anomaly_timestamp')

# trace aggregate earliest
if 'earliest_anomaly_timestamp' in trace_agg.columns:
    trace_agg['earliest_ts_dt'] = parse_tscol(trace_agg, 'earliest_anomaly_timestamp')

# segments start
if 'segment_start_timestamp' in segments_df.columns:
    segments_df['segment_start_dt'] = parse_tscol(segments_df, 'segment_start_timestamp')

# 2) For each unique cmdb_id present in any table compute required aggregations
cmdb_ids = pd.Index(
    list(metric_agg['cmdb_id'].dropna().unique()) +
    list(segments_df['cmdb_id'].dropna().unique()) +
    list(trace_agg['cmdb_id'].dropna().unique())
).unique()

rows = []
for cmdb in cmdb_ids:
    # earliest_anomaly_timestamp: min across metric_agg, trace_agg, segments
    ts_candidates = []
    if 'earliest_ts_dt' in metric_agg.columns:
        ts_candidates += list(metric_agg.loc[metric_agg['cmdb_id'] == cmdb, 'earliest_ts_dt'].dropna().tolist())
    if 'earliest_ts_dt' in trace_agg.columns:
        ts_candidates += list(trace_agg.loc[trace_agg['cmdb_id'] == cmdb, 'earliest_ts_dt'].dropna().tolist())
    if 'segment_start_dt' in segments_df.columns:
        ts_candidates += list(segments_df.loc[segments_df['cmdb_id'] == cmdb, 'segment_start_dt'].dropna().tolist())
    earliest_ts = min(ts_candidates) if ts_candidates else pd.NaT

    # total_anomaly_count:
    # - sum anomaly_count from metric_agg for this cmdb
    m_count = metric_agg.loc[metric_agg['cmdb_id'] == cmdb, 'anomaly_count'].astype(float).sum() if 'anomaly_count' in metric_agg.columns else 0.0
    # - sum anomaly_count from trace_agg
    t_count = trace_agg.loc[trace_agg['cmdb_id'] == cmdb, 'anomaly_count'].astype(float).sum() if 'anomaly_count' in trace_agg.columns else 0.0
    # - sum segment_length_minutes from segments_df
    s_count = segments_df.loc[segments_df['cmdb_id'] == cmdb, 'segment_length_minutes'].astype(float).sum() if 'segment_length_minutes' in segments_df.columns else 0.0

    total_anomaly_count = float(m_count) + float(t_count) + float(s_count)

    # total_severity:
    # - sum severity_value from segments
    seg_sev = segments_df.loc[segments_df['cmdb_id'] == cmdb, 'severity_value'].astype(float).sum() if 'severity_value' in segments_df.columns else 0.0
    # - sum trace proxy severity = max_value_in_window - P95 for each trace row
    trace_rows = trace_agg.loc[trace_agg['cmdb_id'] == cmdb].copy()
    if not trace_rows.empty:
        trace_rows['P95'] = pd.to_numeric(trace_rows['P95'], errors='coerce').fillna(0.0)
        trace_rows['max_value_in_window'] = pd.to_numeric(trace_rows['max_value_in_window'], errors='coerce').fillna(0.0)
        trace_rows['trace_sev_proxy'] = (trace_rows['max_value_in_window'] - trace_rows['P95']).astype(float)
        trace_sev = trace_rows['trace_sev_proxy'].sum()
    else:
        trace_sev = 0.0

    # Optionally include metric aggregate proxy severity for metric_agg where no segments exist?
    # The instruction only required segments' severity and trace proxies; we follow that.
    total_severity = float(seg_sev) + float(trace_sev)

    # faulty_item_count: distinct kpi_name or trace_name flagged as anomalous for this cmdb
    metric_items = set(metric_agg.loc[metric_agg['cmdb_id'] == cmdb, 'kpi_name'].dropna().unique()) if 'kpi_name' in metric_agg.columns else set()
    segment_items = set(segments_df.loc[segments_df['cmdb_id'] == cmdb, 'kpi_name'].dropna().unique()) if 'kpi_name' in segments_df.columns else set()
    trace_items = set(trace_agg.loc[trace_agg['cmdb_id'] == cmdb, 'trace_name'].dropna().unique()) if 'trace_name' in trace_agg.columns else set()
    faulty_items = metric_items.union(segment_items).union(trace_items)
    faulty_item_count = len(faulty_items)

    # example_top_kpi_or_trace: choose item with highest local contribution:
    # build item-level contributions: for segments, sum severity per kpi; for traces, proxy severity; for metric_agg fallback, use anomaly_count
    item_contrib = {}

    # segments severity per kpi
    if 'kpi_name' in segments_df.columns and not segments_df.loc[segments_df['cmdb_id']==cmdb].empty:
        seg_grp = segments_df.loc[segments_df['cmdb_id']==cmdb].groupby('kpi_name', as_index=False).agg(seg_severity_sum=('severity_value','sum'), seg_length_sum=('segment_length_minutes','sum'))
        for _, r in seg_grp.iterrows():
            item_contrib[r['kpi_name']] = {'severity': float(r['seg_severity_sum']), 'anomaly_count': float(r['seg_length_sum'])}

    # trace proxies per trace_name
    if not trace_rows.empty:
        tr_grp = trace_rows.groupby('trace_name', as_index=False).agg(trace_sev_proxy=('trace_sev_proxy','sum'), trace_anom_count=('anomaly_count','sum'))
        for _, r in tr_grp.iterrows():
            item_contrib[r['trace_name']] = {'severity': float(r['trace_sev_proxy']), 'anomaly_count': float(r['trace_anom_count'])}

    # metric agg fallback (kpi names with anomaly_count)
    if 'kpi_name' in metric_agg.columns and not metric_agg.loc[metric_agg['cmdb_id']==cmdb].empty:
        mg = metric_agg.loc[metric_agg['cmdb_id']==cmdb].groupby('kpi_name', as_index=False).agg(m_anom_count=('anomaly_count','sum'))
        for _, r in mg.iterrows():
            name = r['kpi_name']
            if name in item_contrib:
                item_contrib[name]['anomaly_count'] = item_contrib[name].get('anomaly_count',0.0) + float(r['m_anom_count'])
            else:
                item_contrib[name] = {'severity': 0.0, 'anomaly_count': float(r['m_anom_count'])}

    # choose example item: prefer highest severity, tie-breaker by anomaly_count
    example_name = None
    if item_contrib:
        items_df = pd.DataFrame([
            {'name': k, 'severity': v.get('severity',0.0), 'anomaly_count': v.get('anomaly_count',0.0)}
            for k,v in item_contrib.items()
        ])
        items_df = items_df.sort_values(['severity','anomaly_count'], ascending=[False, False]).reset_index(drop=True)
        example_name = items_df.iloc[0]['name']
    else:
        example_name = None

    # determine level
    if isinstance(cmdb, str):
        if cmdb.startswith('os_'):
            level = 'node'
        elif cmdb.startswith('docker_'):
            level = 'pod'
        elif cmdb.startswith('db_') or cmdb.startswith('redis_'):
            level = 'service'
        else:
            level = 'unknown'
    else:
        level = 'unknown'

    rows.append({
        'level': level,
        'cmdb_id': cmdb,
        'faulty_item_count': faulty_item_count,
        'total_anomaly_count': int(total_anomaly_count),
        'earliest_anomaly_timestamp': (earliest_ts.tz_convert(tz).strftime('%Y-%m-%dT%H:%M:%S%z') if pd.notnull(earliest_ts) else None),
        'total_severity': float(total_severity),
        'example_top_kpi_or_trace': example_name
    })

impact_df = pd.DataFrame(rows)

# 4) Return compact table top 20 sorted by total_severity desc then total_anomaly_count desc
impact_top20 = impact_df.sort_values(['total_severity','total_anomaly_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# Keep output compact: selected columns only
impact_top20 = impact_top20.loc[:, ['level','cmdb_id','faulty_item_count','total_anomaly_count','earliest_anomaly_timestamp','total_severity','example_top_kpi_or_trace']]

impact_top20

```
Out[5]:
```
```python
# Compose a concise plain-English summary from the in-memory impact_top20 table.
# Display the summary string variable as the final output (IPython will show it).

try:
    impact_top20
except NameError:
    summary = "No impact table found in kernel."
else:
    # Use top rows to build a short human-readable summary
    top_rows = impact_top20.head(5)
    lines = []
    lines.append("Per-component impact ranking (top results):")
    for i, r in top_rows.iterrows():
        lines.append(
            f"{i+1}. {r['level'].capitalize()} {r['cmdb_id']}: "
            f"{int(r['faulty_item_count'])} faulty items, {int(r['total_anomaly_count'])} total anomaly points, "
            f"earliest={r['earliest_anomaly_timestamp']}, total_severity={r['total_severity']:.5f}, "
            f"example={r['example_top_kpi_or_trace']}"
        )
    lines.append(
        "Interpretation: db_001 (service) shows the highest impact (largest total_severity and anomaly count). "
        "Several docker pods (docker_001/003/002) show notable trace anomalies (JDBC-related traces). "
        "Some nodes/services (os_007, os_020, redis_011/012) have long continuous low-value metric segments but near-zero severity."
    )
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

level     cmdb_id  faulty_item_count  total_anomaly_count earliest_anomaly_timestamp  total_severity      example_top_kpi_or_trace
0  service      db_001                 11                  497   2020-04-11T02:00:00+0800        0.010000                           ACS
1      pod  docker_001                  7                  210   2020-04-11T02:00:00+0800        0.003705    trace.self.jdbc.error_rate
2      pod  docker_003                  6                  180   2020-04-11T02:00:00+0800        0.002311    trace.self.jdbc.error_rate
3      pod  docker_002                  6                  180   2020-04-11T02:00:00+0800        0.000477    trace.self.jdbc.error_rate
4  service      db_002                  5                  150   2020-04-11T02:00:00+0800        0.000000                           ACS
5     node      os_007                  1                   62   2020-04-11T02:00:00+0800        0.000000                   Disk_rd_ios
6     node      os_020                  1                   62   2020-04-11T02:00:00+0800        0.000000                  Disk_io_util
7  service   redis_011                  1                   62   2020-04-11T02:00:00+0800        0.000000                    redis_ping
8  service   redis_012                  1                   62   2020-04-11T02:00:00+0800        0.000000                    redis_load
9      pod  docker_007                  1                   30   2020-04-11T02:00:00+0800        0.000000  trace.self.jdbc.duration_p95```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing in-memory variables if present
g = globals()

# Prefer df_tw (windowed trace merged with thresholds & is_anomaly) if available
if 'df_tw' in g:
    df_tw_local = g['df_tw'].copy()
else:
    # fallback: use df_trace and trace_thresholds to build windowed merged df_tw_local
    if 'df_trace' not in g or 'trace_thresholds' not in g:
        raise RuntimeError("Required in-memory variables `df_trace` and `trace_thresholds` not found.")
    df_trace_local = g['df_trace'].copy()
    trace_thresholds = g['trace_thresholds'].copy()
    # ensure ts column exists
    if 'ts' not in df_trace_local.columns:
        df_trace_local['ts'] = pd.to_datetime(df_trace_local['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # window mask (inclusive)
    start = pd.Timestamp('2020-04-11 02:00:00').tz_localize(tz)
    end = pd.Timestamp('2020-04-11 02:30:00').tz_localize(tz)
    mask_window = (df_trace_local['ts'] >= start) & (df_trace_local['ts'] <= end)
    df_trace_window = df_trace_local[mask_window].copy()
    # merge thresholds and mark anomalies
    df_tw_local = df_trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
    df_tw_local['is_anomaly'] = (df_tw_local['value'] >= df_tw_local['P95']) | (df_tw_local['value'] <= df_tw_local['P5'])

# 1) Filter to trace_name containing 'jdbc' and within the incident window
mask_jdbc = df_tw_local['trace_name'].str.contains('jdbc', case=False, na=False)
df_jdbc = df_tw_local[mask_jdbc].copy()

# If ts column absent, ensure it exists
if 'ts' not in df_jdbc.columns:
    df_jdbc['ts'] = pd.to_datetime(df_jdbc['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 2) Aggregate per (cmdb_id, trace_name)
if df_jdbc.empty:
    jdbc_summary = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','mean_value_in_window','global_P95','global_P5','error_rate_anomaly_count'
    ])
else:
    # compute group aggregates
    grp = df_jdbc.groupby(['cmdb_id','trace_name'], as_index=False)
    agg = grp.agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anom_ts_raw=('ts', lambda s: s[df_jdbc.loc[s.index, 'is_anomaly']].min() if any(df_jdbc.loc[s.index, 'is_anomaly']) else pd.NaT),
        max_value_in_window=('value', 'max'),
        mean_value_in_window=('value', 'mean'),
        P95=('P95', 'first'),
        P5=('P5', 'first')
    )

    # format earliest timestamp strings in Asia/Shanghai
    agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anom_ts_raw'], utc=True, errors='coerce').dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    # replace 'NaT' string results with None
    agg.loc[agg['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None

    # error_rate_anomaly_count: if trace_name contains 'error_rate' then anomaly_count else 0
    agg['error_rate_anomaly_count'] = np.where(agg['trace_name'].str.contains('error_rate', case=False, na=False), agg['anomaly_count'], 0)

    # rename global P95/P5 columns
    agg = agg.rename(columns={'P95':'global_P95','P5':'global_P5'})

    # select and order columns
    jdbc_summary = agg.loc[:, [
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','mean_value_in_window','global_P95','global_P5','error_rate_anomaly_count'
    ]]

    # sort by anomaly_count desc, limit to 20
    jdbc_summary = jdbc_summary.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)

# 3) Cross-reference row for db_001 from metric aggregates (use metric aggregate variable available)
# Prefer metric aggregate variable names that likely exist
metric_candidates = ['anomalies_top20', 'metric_agg', 'anom_combined', 'metric_anom_agg']
metric_df = None
for name in metric_candidates:
    if name in g:
        metric_df = g[name]
        break

if metric_df is None:
    # fallback to impact_top20 if it contains db_001 info
    metric_df = g.get('impact_top20', pd.DataFrame())

# Attempt to extract earliest metric anomaly timestamp and distinct KPI count for db_001
db1_row = {'db_cmdb_id': 'db_001', 'earliest_metric_anomaly_timestamp': None, 'distinct_faulty_kpi_count': 0}
if metric_df is not None and not metric_df.empty:
    # Ensure it's a DataFrame
    metric_df_local = metric_df.copy()
    # If metric_df has 'cmdb_id' and 'kpi_name', proceed
    if 'cmdb_id' in metric_df_local.columns and 'kpi_name' in metric_df_local.columns:
        m_db = metric_df_local[metric_df_local['cmdb_id'] == 'db_001'].copy()
        if not m_db.empty:
            # parse earliest timestamps if column exists
            if 'earliest_anomaly_timestamp' in m_db.columns:
                ts_parsed = pd.to_datetime(m_db['earliest_anomaly_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
                if not ts_parsed.dropna().empty:
                    db1_row['earliest_metric_anomaly_timestamp'] = ts_parsed.min().strftime('%Y-%m-%dT%H:%M:%S%z')
            # distinct faulty KPI count from kpi_name values
            db1_row['distinct_faulty_kpi_count'] = int(m_db['kpi_name'].nunique())
    else:
        # try impact_top20 which already had aggregated faulty count & earliest ts
        if 'impact_top20' in g:
            imp = g['impact_top20']
            if isinstance(imp, pd.DataFrame) and not imp.empty:
                db_imp = imp[imp['cmdb_id']=='db_001']
                if not db_imp.empty:
                    db1_row['earliest_metric_anomaly_timestamp'] = db_imp.iloc[0]['earliest_anomaly_timestamp']
                    db1_row['distinct_faulty_kpi_count'] = int(db_imp.iloc[0]['faulty_item_count'])

# Assemble cross-reference as a one-row DataFrame
cross_ref_db1 = pd.DataFrame([db1_row])

# Return the compact aggregates: jdbc_summary (top up to 20) and cross_ref_db1
jdbc_summary, cross_ref_db1

```
Out[6]:
```
Summary of JDBC-related trace findings (window 2020-04-11 02:00→02:30 +08):

- Scope: JDBC traces from many pods (docker_001 through docker_008) show anomalies during the window. The returned compact table includes 20 JDBC-related trace rows (examples below).

- Continuous per-minute anomalies:
  - Several pods have anomaly_count = 30 (i.e., every minute in the 30-minute window). Notably:
    - docker_001: trace.self.jdbc.duration_p95 — 30 anomalies (P95=0.003, max=0.004, mean≈0.00277)
    - docker_001: trace.self.jdbc.error_rate — 30 anomalies (P95=0.000, max≈0.002705, error_rate_anomaly_count=30)
    - docker_002: trace.self.jdbc.error_rate — 30 anomalies (P95≈0.000466, max≈0.000943, error_rate_anomaly_count=30)
    - docker_003, docker_004, docker_005, docker_006, docker_007, docker_008 likewise show JDBC duration_p95 and/or error_rate traces flagged every minute.
  - For many error_rate traces the global P95/P5 are 0 and max/min are very small; nonetheless every point met the anomaly criterion.

- Intermittent spikes / additional JDBC signals:
  - docker_002: trace.self.jdbc.duration_p95 — 9 anomalies starting at 2020-04-11T02:09:00+08, max=0.013 (larger spike vs typical).
  - docker_003 and docker_004 show short (4-point) anomalies for trace.self.jdbc.duration_mean and trace.self.jdbc.row_count with large row_count maxima (e.g., docker_004 row_count max ≈2529).

- Error-rate specific count:
  - For trace names containing "error_rate", error_rate_anomaly_count equals anomaly_count (many = 30), indicating persistent error_rate anomalies across pods.

- Cross-reference to DB metrics (db_001):
  - Earliest metric anomaly for db_001: 2020-04-11T02:00:00+08.
  - Number of distinct db_001 KPIs flagged as faulty: 11.

Interpretation / takeaway (concise):
- JDBC-related trace anomalies are widespread across pods throughout the entire incident window. Many pods have persistent per-minute anomalies in jdbc error_rate traces; a few pods show short duration spikes in JDBC latency/row_count (e.g., docker_002, docker_003, docker_004).
- db_001 metric anomalies already appear at 02:00 and involve many KPIs, temporally aligning with the trace anomalies. Recommend prioritizing investigation of pod-level JDBC error_rate and the db_001 service (check JDBC connections, client errors, and pod logs around 02:00–02:30).

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                     trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  mean_value_in_window   global_P95    global_P5  error_rate_anomaly_count
0   docker_001   trace.self.jdbc.duration_p95             30   2020-04-11T02:00:00+0800             0.004000              0.002767     0.003000     0.001000                         0
1   docker_001     trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800             0.002705              0.000090     0.000000     0.000000                        30
2   docker_002     trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800             0.000943              0.000060     0.000466     0.000000                        30
3   docker_003   trace.self.jdbc.duration_p95             30   2020-04-11T02:00:00+0800             0.004000              0.002767     0.003000     0.001000                         0
4   docker_004   trace.self.jdbc.duration_p95             30   2020-04-11T02:00:00+0800             0.006000              0.002833     0.003000     0.001000                         0
5   docker_004     trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800             0.001702              0.000091     0.000560     0.000000                        30
6   docker_003     trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800             0.001311              0.000044     0.000000     0.000000                        30
7   docker_005   trace.self.jdbc.duration_p95             30   2020-04-11T02:00:00+0800             0.003000              0.003000     0.003000     0.003000                         0
8   docker_007   trace.self.jdbc.duration_p95             30   2020-04-11T02:00:00+0800             0.003000              0.003000     0.003000     0.003000                         0
9   docker_007     trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800             0.000000              0.000000     0.000000     0.000000                        30
10  docker_008     trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800             0.000000              0.000000     0.000000     0.000000                        30
11  docker_008   trace.self.jdbc.duration_p95             30   2020-04-11T02:00:00+0800             0.003000              0.003000     0.003000     0.003000                         0
12  docker_006   trace.self.jdbc.duration_p95             30   2020-04-11T02:00:00+0800             0.003000              0.003000     0.003000     0.003000                         0
13  docker_006     trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800             0.000000              0.000000     0.000000     0.000000                        30
14  docker_005     trace.self.jdbc.error_rate             30   2020-04-11T02:00:00+0800             0.000000              0.000000     0.000000     0.000000                        30
15  docker_002   trace.self.jdbc.duration_p95              9   2020-04-11T02:09:00+0800             0.013000              0.003200     0.004000     0.001000                         0
16  docker_004      trace.self.jdbc.row_count              4   2020-04-11T02:07:00+0800          2529.000000           2161.500000  2466.600000  1852.500000                         0
17  docker_004  trace.self.jdbc.duration_mean              4   2020-04-11T02:04:00+0800             0.002662              0.001745     0.002331     0.001183                         0
18  docker_003  trace.self.jdbc.duration_mean              4   2020-04-11T02:12:00+0800             0.002875              0.002005     0.002488     0.001629                         0
19  docker_003      trace.self.jdbc.row_count              4   2020-04-11T02:06:00+0800          2658.000000           2231.033333  2548.100000  1915.100000                         0,   db_cmdb_id earliest_metric_anomaly_timestamp  distinct_faulty_kpi_count
0     db_001          2020-04-11T02:00:00+0800                         11)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric if present, otherwise load
if 'df_metric' in globals():
    dfm = df_metric.copy()
else:
    dfm = pd.read_csv('metric.csv')
    dfm['ts'] = pd.to_datetime(dfm['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure ts column exists and is timezone-aware Asia/Shanghai
if 'ts' not in dfm.columns:
    dfm['ts'] = pd.to_datetime(dfm['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # normalize tz to Asia/Shanghai if not already
    dfm['ts'] = pd.to_datetime(dfm['ts'], utc=True, errors='coerce').dt.tz_convert(tz)

# 1) Compute global P95 and P5 for every kpi_name belonging to cmdb_id 'db_001' using full series
df_db_all = dfm[dfm['cmdb_id'] == 'db_001'].copy()

# If no rows for db_001, produce empty results
if df_db_all.empty:
    db001_kpi_summary = pd.DataFrame(columns=[
        'kpi_name','anomaly_count','earliest_anomaly_timestamp','P95','P5',
        'max_in_window','min_in_window','mean_in_window','severity'
    ])
    db001_overall_summary = pd.DataFrame([{
        'earliest_metric_anomaly_timestamp': None,
        'distinct_faulty_kpi_count': 0,
        'total_severity_sum': 0.0
    }])
else:
    thresholds_db = (
        df_db_all
        .groupby('kpi_name', as_index=False)['value']
        .agg(P95=lambda s: s.quantile(0.95), P5=lambda s: s.quantile(0.05))
    )

    # 2) Filter db_001 rows to the incident window (inclusive)
    start = pd.Timestamp('2020-04-11 02:00:00').tz_localize(tz)
    end = pd.Timestamp('2020-04-11 02:30:00').tz_localize(tz)
    df_db_win = df_db_all[(df_db_all['ts'] >= start) & (df_db_all['ts'] <= end)].copy()

    # Merge thresholds
    df_db_win = df_db_win.merge(thresholds_db, on='kpi_name', how='left')

    # mark anomalies
    df_db_win['is_anomaly'] = (df_db_win['value'] >= df_db_win['P95']) | (df_db_win['value'] <= df_db_win['P5'])

    # group by kpi_name to compute required metrics
    grp = df_db_win.groupby('kpi_name', as_index=False)
    agg = grp.agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anom_ts_raw=('ts', lambda s: s[df_db_win.loc[s.index, 'is_anomaly']].min() if any(df_db_win.loc[s.index, 'is_anomaly']) else pd.NaT),
        max_in_window=('value', 'max'),
        min_in_window=('value', 'min'),
        mean_in_window=('value', 'mean'),
        P95=('P95', 'first'),
        P5=('P5', 'first')
    )

    # compute severity per kpi: high = max - P95, low = P5 - min; choose larger magnitude (and non-negative)
    agg['high_sev'] = (agg['max_in_window'] - agg['P95']).clip(lower=0)
    agg['low_sev'] = (agg['P5'] - agg['min_in_window']).clip(lower=0)
    agg['severity'] = agg[['high_sev','low_sev']].max(axis=1)

    # format earliest_anom_ts
    agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anom_ts_raw'], utc=True, errors='coerce').dt.tz_convert(tz).dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    agg.loc[agg['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None

    # select and order columns
    db001_kpi_summary = agg.loc[:, [
        'kpi_name','anomaly_count','earliest_anomaly_timestamp','P95','P5',
        'max_in_window','min_in_window','mean_in_window','severity'
    ]].copy()

    # numeric formatting: keep reasonable decimals
    for c in ['P95','P5','max_in_window','min_in_window','mean_in_window','severity']:
        db001_kpi_summary[c] = pd.to_numeric(db001_kpi_summary[c], errors='coerce')

    # limit to 50 rows
    db001_kpi_summary = db001_kpi_summary.sort_values(['anomaly_count','severity'], ascending=[False, False]).head(50).reset_index(drop=True)

    # 3) overall summary row for db_001
    earliest_ts_vals = pd.to_datetime(db001_kpi_summary['earliest_anomaly_timestamp'], utc=True, errors='coerce').dropna()
    earliest_metric_anomaly_timestamp = earliest_ts_vals.min().tz_convert(tz).strftime('%Y-%m-%dT%H:%M:%S%z') if not earliest_ts_vals.empty else None
    distinct_faulty_kpi_count = int((db001_kpi_summary['anomaly_count'] > 0).sum())
    total_severity_sum = float(db001_kpi_summary['severity'].sum())

    db001_overall_summary = pd.DataFrame([{
        'earliest_metric_anomaly_timestamp': earliest_metric_anomaly_timestamp,
        'distinct_faulty_kpi_count': distinct_faulty_kpi_count,
        'total_severity_sum': total_severity_sum
    }])

# Display compact results
db001_kpi_summary, db001_overall_summary

```
Out[7]:
```
Summary for db_001 (verification of DB-side symptom pattern):

- Scope: 45 distinct db_001 KPIs were flagged as anomalous in the incident window.
- Timing: the earliest DB metric anomaly occurred at 2020-04-11T02:00:00+08 (i.e., anomalies begin at the very start of the window).
- Persistence: many key DB KPIs show anomalies for the full window (anomaly_count = 30), e.g., MEM_real_util (30 points, P95=46.166, P5=46.150, max=46.280, mean≈46.154, severity=0.114) and ACS (30 points, P95=0.02, P5=0.01, max=0.02, severity=0.01).
- Severity: the summed severity across all db_001 KPIs is large (total_severity_sum = 15593.227056), indicating some KPIs had substantial deviations; many KPIs have zero per-kpi severity because their values equal the stored P95/P5 but still counted as anomalies.
- Interpretation: there is a broad, DB-side symptom set starting at 02:00 affecting many KPIs (45 items). Some metrics show meaningful deviation (e.g., MEM_real_util, ACS), while many others are at threshold values (leading to long continuous anomaly counts). 
- Recommendation: prioritize investigation of db_001 immediately (memory/utilization and KPIs with non-zero severity), and correlate these DB-side anomalies with the JDBC trace anomalies in pods around 02:00–02:30.

The original code execution output of IPython Kernel is also provided below for reference:

(          kpi_name  anomaly_count earliest_anomaly_timestamp           P95            P5  max_in_window  min_in_window  mean_in_window  severity
0    MEM_real_util             30   2020-04-11T02:00:00+0800     46.166000     46.150000      46.280000      46.150000       46.154333     0.114
1              ACS             30   2020-04-11T02:00:00+0800      0.020000      0.010000       0.020000       0.000000        0.014667     0.010
2             AIOS             30   2020-04-11T02:00:00+0800      0.010000      0.000000       0.010000       0.000000        0.001000     0.000
3             Hang             30   2020-04-11T02:00:00+0800      1.000000      1.000000       1.000000       1.000000        1.000000     0.000
4        MEM_Total             30   2020-04-11T02:00:00+0800    381.902264    381.902264     381.902264     381.902264      381.902264     0.000
..             ...            ...                        ...           ...           ...            ...            ...             ...       ...
40     Tbs_Free_Gb              1   2020-04-11T02:27:00+0800    551.390000    551.390000     551.390000     551.390000      551.390000     0.000
41    Tbs_Used_Pct              1   2020-04-11T02:27:00+0800     86.200000     86.200000      86.200000      86.200000       86.200000     0.000
42  Total_Tbs_Size              1   2020-04-11T02:27:00+0800  11415.270000  11415.270000   11415.270000   11415.270000    11415.270000     0.000
43     UndoTbs_Pct              1   2020-04-11T02:27:00+0800      2.150000      2.150000       2.150000       2.150000        2.150000     0.000
44   Used_Tbs_Size              1   2020-04-11T02:27:00+0800   9149.660000   9149.660000    9149.660000    9149.660000     9149.660000     0.000

[45 rows x 9 columns],   earliest_metric_anomaly_timestamp  distinct_faulty_kpi_count  total_severity_sum
0          2020-04-11T02:00:00+0800                         45        15593.227056)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```